# 롤 API 데이터 수집하기

In [1]:
import requests
import json
import time
import numpy as np
import pandas as pd

## API를 통한 소환사 정보 수집

In [2]:
# Lol Api에서 소환사 명을 통해서 게임 정보를 알 수 있는 puuid 불러오기 
summoner_Name = 'one summer drive' #소환사명
api_key = "RGAPI-c0b8055b-4074-4950-b279-1feae9cdaf1a" # api key
request_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/103.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Charset": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://developer.riotgames.com",
    "X-Riot-Token": "RGAPI-c0b8055b-4074-4950-b279-1feae9cdaf1a"
}

summoner_url="https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/"+summoner_Name+"/""?api_key="+api_key

In [3]:
# requests를 통해서 puuid 추출하기
def summoner(summoner_Name, api_key):
    summoner_url = "https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/"+summoner_Name+"/""?api_key="+api_key
    return requests.get(summoner_url, headers=request_headers).json()['puuid']
summoner_puuid = summoner(summoner_Name, api_key)
summoner_puuid

'cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72J0XfluBWZsNURa-_rKA5AR_vMndVW8tg'

In [4]:
# match id를 확인할 수 있는 api를 통해서 조회하기
# match_count : 조회하고 싶은 매치의 수
# 더 많이 조회를 하고 싶으나 횟수 제한이 있다. (100) 이번 시즌 내가 플레이한 62게임에 대해서 조사하기 
def match(summoner_puuid, match_count):
    match_url = "https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/"+summoner_puuid+"/ids?start=0&count="+str(match_count)
    return requests.get(match_url, headers=request_headers).json()
match_id = match(summoner_puuid, 90)
match_id

['KR_6027078402',
 'KR_6026959324',
 'KR_6026550378',
 'KR_6026049230',
 'KR_6025776447',
 'KR_6025626879',
 'KR_6024617209',
 'KR_6024522598',
 'KR_6024290118',
 'KR_6024206432',
 'KR_6018696284',
 'KR_6018287979',
 'KR_6017207879',
 'KR_6017033024',
 'KR_6016828310',
 'KR_6016772762',
 'KR_6016770544',
 'KR_6016651859',
 'KR_6016575318',
 'KR_6015787618',
 'KR_6015488822',
 'KR_6015088138',
 'KR_6014287649',
 'KR_6012864614',
 'KR_6012476838',
 'KR_6011712595',
 'KR_6011597486',
 'KR_6010856241',
 'KR_6009908963',
 'KR_6009876008',
 'KR_6009892943',
 'KR_6009828517',
 'KR_6009775487',
 'KR_6009753833',
 'KR_6008388001',
 'KR_6008101825',
 'KR_6007718428',
 'KR_6007632404',
 'KR_6007543140',
 'KR_6007407898',
 'KR_6007030534',
 'KR_6006751091',
 'KR_6006310180',
 'KR_6006082114',
 'KR_6005589922',
 'KR_6005521617',
 'KR_6005292412',
 'KR_6004906534',
 'KR_6002949666',
 'KR_6002335329',
 'KR_6000663825',
 'KR_6000095173',
 'KR_5995456456',
 'KR_5993880535',
 'KR_5992325428',
 'KR_59922

In [5]:
# 게임 세부내용 불러오기
# 조회는 하나씩만 가능하므로 for문이나 while문을 돌려 하나씩 조회한다.
def game(match_id_num):
    game_url = "https://asia.api.riotgames.com/lol/match/v5/matches/"+match_id_num
    return requests.get(game_url, headers=request_headers).json()['info']['participants']


## 분석에 사용할 90개의 데이터 수집하여 데이터 프레임 생성
- 100개로 하고 돌렸으나 API응답문제가 발생하였다

In [6]:
#Future Warning 메세지 제거하기
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# 게임 데이터를 담을 DataFrame 생성하기
game_df = pd.DataFrame(columns=['puuid', 'teamPosition', 'individualPosition', 'champName', 'champExp', 'summoner_spell1', 'summoner_spell2' ,
                                'kills', 'deaths', 'assists', 'damageDealtToBuildings', 'totalDamageDealtToChamp', 'damagePerMin', 'teamDamagePer(%)',
                                'killParticipation(%)', 'totalDamageTaken', 'damageTakeOnTeamPer(%)', 'goldEarned', 'goldPerMin', 'totalCs',
                                'maxCsAdvantageOnLaneOpponent', 'maxLevelLeadLaneOpponent', 'gameEndedInEarlySurren', 'gameEndedInSurren',
                                'teamEarlySurrn', 'timePlayed'])

for i in range(len(match_id)):
    print(i)
    game_content = game(match_id[i])
    for j in range(len(game_content)):
        try:
            input_data = {

                'puuid':game_content[j]['puuid'], 
                'teamPosition':game_content[j]['teamPosition'], 
                'individualPosition':game_content[j]['individualPosition'], 
                'champName':game_content[j]['championName'], 
                'champExp':float(game_content[j]['champExperience']), 
                'summoner_spell1':float(game_content[j]['summoner1Id']), 
                'summoner_spell2':float(game_content[j]['summoner2Id']),
                'kills':float(game_content[j]['kills']), 
                'deaths':float(game_content[j]['deaths']), 
                'assists':float(game_content[j]['assists']), 
                'damageDealtToBuildings':float(game_content[j]['damageDealtToBuildings']), 
                'totalDamageDealtToChamp':float(game_content[j]['totalDamageDealtToChampions']),
                'damagePerMin':game_content[j]['challenges']['damagePerMinute'], 
                'teamDamagePer(%)':round(game_content[j]['challenges']['teamDamagePercentage'], 2)*100,
                'killParticipation(%)':game_content[j]['challenges']['killParticipation']*100,
                'totalDamageTaken':float(game_content[j]['totalDamageTaken']),
                'damageTakeOnTeamPer(%)':round(game_content[j]['challenges']['damageTakenOnTeamPercentage'],2)*100,
                'goldEarned':float(game_content[j]['goldEarned']),
                'goldPerMin':game_content[j]['challenges']['goldPerMinute'],
                'totalCs':float(game_content[j]['totalMinionsKilled']+game_content[j]['neutralMinionsKilled']),
                'maxCsAdvantageOnLaneOpponent':float(game_content[j]['challenges']['maxCsAdvantageOnLaneOpponent']),
                'maxLevelLeadLaneOpponent':float(game_content[j]['challenges']['maxLevelLeadLaneOpponent']),
                'gameEndedInEarlySurren':game_content[j]['gameEndedInEarlySurrender'],
                'gameEndedInSurren':game_content[j]['gameEndedInSurrender'],
                'teamEarlySurrn':game_content[j]['teamEarlySurrendered'],
                'timePlayed':float(round(game_content[j]['timePlayed']/60,2)*100) # 4자리로 앞의 2자리는 분, 뒤의 2자리는 초로 구성되어있다. 
                }
            game_df = game_df.append(input_data, ignore_index=True)
        except:
            input_data = {

                'puuid':game_content[j]['puuid'], 
                'teamPosition':game_content[j]['teamPosition'], 
                'individualPosition':game_content[j]['individualPosition'], 
                'champName':game_content[j]['championName'], 
                'champExp':float(game_content[j]['champExperience']), 
                'summoner_spell1':float(game_content[j]['summoner1Id']), 
                'summoner_spell2':float(game_content[j]['summoner2Id']),
                'kills':float(game_content[j]['kills']), 
                'deaths':float(game_content[j]['deaths']), 
                'assists':float(game_content[j]['assists']), 
                'damageDealtToBuildings':float(game_content[j]['damageDealtToBuildings']), 
                'totalDamageDealtToChamp':float(game_content[j]['totalDamageDealtToChampions']),
                'damagePerMin':game_content[j]['challenges']['damagePerMinute'], 
                'teamDamagePer(%)':round(game_content[j]['challenges']['teamDamagePercentage'], 2)*100,
                'killParticipation(%)':game_content[j]['challenges']['killParticipation']*100,
                'totalDamageTaken':float(game_content[j]['totalDamageTaken']),
                'damageTakeOnTeamPer(%)':round(game_content[j]['challenges']['damageTakenOnTeamPercentage'],2)*100,
                'goldEarned':float(game_content[j]['goldEarned']),
                'goldPerMin':game_content[j]['challenges']['goldPerMinute'],
                'totalCs':float(game_content[j]['totalMinionsKilled']+game_content[j]['neutralMinionsKilled']),
                'maxCsAdvantageOnLaneOpponent':0,
                'maxLevelLeadLaneOpponent':0,
                'gameEndedInEarlySurren':game_content[j]['gameEndedInEarlySurrender'],
                'gameEndedInSurren':game_content[j]['gameEndedInSurrender'],
                'teamEarlySurrn':game_content[j]['teamEarlySurrendered'],
                'timePlayed':float(round(game_content[j]['timePlayed']/60,2)*100) # 4자리로 앞의 2자리는 분, 뒤의 2자리는 초로 구성되어있다. 
                }
            game_df = game_df.append(input_data, ignore_index=True) 

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89


In [7]:
game_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   puuid                         900 non-null    object 
 1   teamPosition                  900 non-null    object 
 2   individualPosition            900 non-null    object 
 3   champName                     900 non-null    object 
 4   champExp                      900 non-null    float64
 5   summoner_spell1               900 non-null    float64
 6   summoner_spell2               900 non-null    float64
 7   kills                         900 non-null    float64
 8   deaths                        900 non-null    float64
 9   assists                       900 non-null    float64
 10  damageDealtToBuildings        900 non-null    float64
 11  totalDamageDealtToChamp       900 non-null    float64
 12  damagePerMin                  900 non-null    float64
 13  teamD

In [8]:
game_df

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,inhHqhkSY41yUqZBdTrfQzJNvLVDSTc3NP0CJ-pMXCWECi...,TOP,TOP,Kennen,15840.0,12.0,4.0,7.0,4.0,15.0,...,17.0,13200.0,425.212970,154.0,8.0,2.0,False,False,False,3103.0
1,421ERaCTa3XOWL-B37zDDq-fUgrHMka2nvlCSz04X90PD6...,JUNGLE,JUNGLE,Warwick,16570.0,11.0,4.0,9.0,6.0,16.0,...,33.0,13733.0,442.358738,178.0,4.0,1.0,False,False,False,3103.0
2,3mf9yiACQN5B6KqQcYiH5L4BHd0mxzm57LYwTf7BepcW74...,MIDDLE,MIDDLE,Zoe,15822.0,4.0,14.0,6.0,5.0,13.0,...,13.0,12318.0,396.790523,197.0,35.0,1.0,False,False,False,3103.0
3,nw03aFpUmsC5SMigTAjfFem779ViT1twLf-otdfWRn4WZk...,BOTTOM,BOTTOM,Samira,14743.0,7.0,4.0,16.0,5.0,10.0,...,18.0,16028.0,516.303129,223.0,7.0,2.0,False,False,False,3103.0
4,fzMBWWb-1HOBVc18kl8RywiuBVlcX9ImqtapnaaAOJCTQH...,UTILITY,UTILITY,Pyke,12659.0,3.0,4.0,5.0,4.0,15.0,...,19.0,9854.0,317.418470,46.0,27.0,1.0,False,False,False,3103.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,HDySFbEK8I7RLt92G5Dy32IaYLIUd8rVrLL2wtkwDTxFKU...,,Invalid,Morgana,19782.0,4.0,32.0,9.0,10.0,28.0,...,21.0,13294.0,669.336182,16.0,1.0,2.0,False,False,False,1985.0
896,8TwMcCi77kqbKIodPvBjOTa2k1_7wGcv0TUEZ6UxYHQTiC...,,Invalid,Yone,18866.0,32.0,4.0,5.0,12.0,14.0,...,25.0,11341.0,571.015660,17.0,0.0,2.0,False,False,False,1985.0
897,MEU7XXrspQCwaBEwxgsxkgJ1-l9SDI5yF0WLUsAtTVjr6O...,,Invalid,Kayn,14886.0,1.0,13.0,6.0,12.0,10.0,...,18.0,11899.0,599.116945,8.0,0.0,0.0,False,False,False,1985.0
898,yomjFh9W-di7bIqnMjRs1AMWpGJvuItPlLyyX6q9y41BIj...,,Invalid,Caitlyn,18334.0,1.0,4.0,21.0,9.0,13.0,...,17.0,17568.0,884.497507,79.0,41.0,2.0,False,False,False,1985.0


## 가설 확인하기

```
인게임 속의 트롤링은 처음부터 발생할 수도 있고 중간에 소환사의 변심으로 인해서 발생할 수도 있고 수 많은 경우로 인해 발생한다.
기본적으로 의도적인 죽음, 다른 소환사들 방해, 한타에 합류를 하지 않거나, 우물에서 잠수를 하거나 
위와 같은 행위들은 기본적으로 DPM의 하락을 가져온다. 또한 안정적으로 성장을 한 상대 라이너에 비해서 성장 측면에서 많은 차이가 나타난다. 

픽창에서의 싸움으로 인한 소환사 스펠, 주 포지션이 아니므로 인한 트롤 등에 대해서는 소환사 스펠을 통해서 확인한다. 
```

- 가장 최근 게임의 트롤은 워익이었다. 
- 욕설과 함께 한타참여X, 정글링만 하거나, 우물에서 잠수 
- 딜량에 대해서 수치화 하기 
- 게임은 빠른 서렌이 나오는 15분부터 끝나기 시작한다.
- 15분까지는 기본적으로 라인전이 끝나는 시기, 탑, 미드, 바텀의 경우에는 라인전을 통해서 딜량을 쌓을수 있으나 정글의 경우에는 힘들다. 
- 게임이 장기간으로 가는경우 서포터는 딜량이 작기 마련이다. 
- 원딜의 경우에는 코어가 뜨면 뜰 수록 강해진다. 
- 위와 같은 경우를 수치화 조정해서 딜량을 계산한다.

## Match에 포함되어 있는 칼바람, 일반게임 정보 제거하기 

In [9]:
# 포지션에 Top, Jungle, Middle, Botoom, Utility 이외의 공백인 행들이 일반게임, 칼바람나락의 결과
game_df['teamPosition'].unique()

array(['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY', ''], dtype=object)

In [10]:
game_df['teamPosition']==''.index

0      False
1      False
2      False
3      False
4      False
       ...  
895    False
896    False
897    False
898    False
899    False
Name: teamPosition, Length: 900, dtype: bool

In [11]:
# 포지션 선택이 없는 게임 제거하기
# 29게임이 제외되었다. 
game_df = game_df.drop(game_df[game_df['teamPosition']==''].index, axis=0).reset_index(drop=True)
game_df

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,inhHqhkSY41yUqZBdTrfQzJNvLVDSTc3NP0CJ-pMXCWECi...,TOP,TOP,Kennen,15840.0,12.0,4.0,7.0,4.0,15.0,...,17.0,13200.0,425.212970,154.0,8.0,2.0,False,False,False,3103.0
1,421ERaCTa3XOWL-B37zDDq-fUgrHMka2nvlCSz04X90PD6...,JUNGLE,JUNGLE,Warwick,16570.0,11.0,4.0,9.0,6.0,16.0,...,33.0,13733.0,442.358738,178.0,4.0,1.0,False,False,False,3103.0
2,3mf9yiACQN5B6KqQcYiH5L4BHd0mxzm57LYwTf7BepcW74...,MIDDLE,MIDDLE,Zoe,15822.0,4.0,14.0,6.0,5.0,13.0,...,13.0,12318.0,396.790523,197.0,35.0,1.0,False,False,False,3103.0
3,nw03aFpUmsC5SMigTAjfFem779ViT1twLf-otdfWRn4WZk...,BOTTOM,BOTTOM,Samira,14743.0,7.0,4.0,16.0,5.0,10.0,...,18.0,16028.0,516.303129,223.0,7.0,2.0,False,False,False,3103.0
4,fzMBWWb-1HOBVc18kl8RywiuBVlcX9ImqtapnaaAOJCTQH...,UTILITY,UTILITY,Pyke,12659.0,3.0,4.0,5.0,4.0,15.0,...,19.0,9854.0,317.418470,46.0,27.0,1.0,False,False,False,3103.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,izkdkzth-nRWiGU1ssUGwGOZSmIx3E0CnVw8TvmCdbADRI...,TOP,TOP,Lucian,10465.0,12.0,4.0,5.0,2.0,3.0,...,21.0,9242.0,512.974362,160.0,63.0,2.0,False,True,False,1800.0
716,GLUb-YiZWPONWpR3wd4E7w17Qmez0-Zowkk-ZmbdchnAAL...,JUNGLE,JUNGLE,Viego,8087.0,11.0,4.0,5.0,4.0,3.0,...,35.0,7629.0,423.453881,97.0,28.0,2.0,False,True,False,1800.0
717,QOU8VObhOsaqbkB3fo8RhBWfhSnBlA6eL6LN5VMP3rl9I3...,MIDDLE,MIDDLE,Lissandra,9822.0,12.0,4.0,7.0,1.0,3.0,...,20.0,8856.0,491.572199,133.0,44.0,2.0,False,True,False,1800.0
718,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Jhin,6736.0,4.0,7.0,1.0,1.0,6.0,...,10.0,7273.0,403.683500,147.0,35.0,1.0,False,True,False,1800.0


### Play시간대 별로 나누기 

#### 20분 이내 종료된 게임

In [12]:
# 5게임만이 61게임중에서 빠르게 끝난 게임 
game_played20down = game_df.copy()[game_df['timePlayed']<2000].reset_index(drop=True)
game_played20down

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,B6f-hh1my-rb1rAAAQTgeJGKVcSXZxZpVPRdWWsUvVeVDO...,TOP,TOP,Nilah,6025.0,4.0,12.0,1.0,3.0,0.0,...,29.0,4026.0,296.781612,88.0,7.00,1.0,False,True,False,1355.0
1,ciPKyG5R0qSB1XvlPWNYaIsnyWDdiBH6Gf6Z2QFw7K1feU...,JUNGLE,JUNGLE,MasterYi,3038.0,11.0,4.0,0.0,7.0,1.0,...,33.0,3081.0,227.123002,35.0,0.00,0.0,False,True,False,1355.0
2,bVKmCzSI3ZRvfE4OZO-IBJ7zol5FjupNBH6kQVmzrH101X...,MIDDLE,MIDDLE,Zyra,6419.0,4.0,12.0,0.0,0.0,0.0,...,5.0,4479.0,330.173516,105.0,7.00,1.0,False,True,False,1355.0
3,Wufvx6Ms2y_bj-zgjjUV4wvOBFU0wpMGg10RRyMQ-7pgd2...,BOTTOM,BOTTOM,Lucian,3766.0,7.0,4.0,1.0,4.0,0.0,...,20.0,4586.0,338.060788,93.0,11.00,0.0,False,True,False,1355.0
4,-dUAnoFX34YOzyqatXBEfUttJfkvkzVK2TbXpQBvs6Y-2A...,UTILITY,UTILITY,Yuumi,3388.0,14.0,3.0,0.0,3.0,1.0,...,13.0,2442.0,180.071371,1.0,0.00,0.0,False,True,False,1355.0
5,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,7597.0,4.0,12.0,3.0,1.0,1.0,...,24.0,5326.0,392.643939,96.0,8.00,2.0,False,True,False,1355.0
6,Dj8_KHWZSrVRcsIMYuLd7Ok9mttISfUy3SlgmovTteTnlV...,JUNGLE,JUNGLE,Amumu,5423.0,11.0,4.0,5.0,1.0,2.0,...,33.0,5272.0,388.655133,84.0,49.00,3.0,False,True,False,1355.0
7,aUu1CsABRqua4d3Wwmh9-cg8w5PLE6u6Y8PG1Wxrf1kzOm...,MIDDLE,MIDDLE,Sylas,7465.0,14.0,4.0,3.0,0.0,0.0,...,21.0,4697.0,346.263318,103.0,7.00,2.0,False,True,False,1355.0
8,HL6hmznifOEV6msIuafQIIrzTtsdJRPCE4KZqck84-UAXP...,BOTTOM,BOTTOM,Jhin,4539.0,7.0,4.0,4.0,0.0,4.0,...,11.0,6170.0,454.865832,98.0,12.00,2.0,False,True,False,1355.0
9,G9HobrTn_m98oKGZeEgjrw2uQbp_HBAc41OsG-RVa6uj7O...,UTILITY,UTILITY,FiddleSticks,4386.0,3.0,4.0,2.0,0.0,5.0,...,11.0,4655.0,343.178177,7.0,5.75,1.0,False,True,False,1355.0


#### 20분 ~ 30분 사이 종료된 게임

In [13]:
# 20분에서 30분 사이 종료된 게임으로는 33게임이 존재한다. 
game_played2030 = game_df.copy()[(2000<= game_df['timePlayed']) & (game_df['timePlayed'] < 3000)].reset_index(drop=True)
game_played2030

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,13406.0,4.0,12.0,3.0,6.0,2.0,...,21.0,11210.0,381.745048,212.0,52.15,3.0,False,True,False,2935.0
1,vC7V3QyBHaKO98rh_PKZOQPh2emLzh-IonReZDAxiWpG1C...,JUNGLE,JUNGLE,Nidalee,9776.0,11.0,4.0,2.0,8.0,3.0,...,29.0,9017.0,307.061024,130.0,4.00,1.0,False,True,False,2935.0
2,b1qulCM6x6M4dLBKm7xE0fmHX6nJOM1UIF9Tvlm4SPyEqR...,MIDDLE,MIDDLE,Tristana,13043.0,14.0,4.0,3.0,6.0,1.0,...,18.0,11303.0,384.892289,230.0,88.50,1.0,False,True,False,2935.0
3,fax7ch_-ejAH7-P0Nevz9q9Wdj7WN32zWfFgu2UYna1Dd-...,BOTTOM,BOTTOM,Jhin,10333.0,7.0,4.0,0.0,5.0,3.0,...,12.0,8455.0,287.928704,187.0,9.00,1.0,False,True,False,2935.0
4,uZ5Zgf2evXDYMRZruzkf66uz5XYZtsEP7XgpYXOHUuKQs6...,UTILITY,UTILITY,Seraphine,7809.0,4.0,3.0,0.0,9.0,5.0,...,20.0,6217.0,211.701769,25.0,0.00,1.0,False,True,False,2935.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365,QxffUyKLIHztu5DPi06OglE9Sf8Gr2NdsNY1d7EhFRfJpe...,TOP,TOP,LeeSin,7324.0,12.0,4.0,3.0,8.0,3.0,...,22.0,6656.0,317.423426,80.0,17.00,1.0,False,True,False,2097.0
366,7laSMnqFaZQgu3zqX_GVtjihk_AykGaA-0xIwjOB_20C1G...,JUNGLE,JUNGLE,Zac,5613.0,11.0,4.0,1.0,9.0,2.0,...,29.0,5155.0,245.845942,66.0,5.00,1.0,False,True,False,2097.0
367,MmWFm-YgIIxp0naNdqhU4hMmW00aI9V8dVM5mCQDdXsipx...,MIDDLE,MIDDLE,Yasuo,9404.0,14.0,4.0,6.0,6.0,4.0,...,23.0,10991.0,524.114813,151.0,65.00,1.0,False,True,False,2097.0
368,tPblUi0-8C2VmSudF5TeoPjvU2ifuXGZfYDHQHGDiirJ7L...,BOTTOM,UTILITY,Jhin,6091.0,7.0,4.0,1.0,8.0,3.0,...,15.0,5339.0,254.619547,70.0,40.00,1.0,False,True,False,2097.0


#### 30분 ~ 40분 사이 종료된 게임

In [14]:
# 20게임이 존재한다.
game_played3040 = game_df.copy()[(3000<= game_df['timePlayed']) & (game_df['timePlayed'] < 4000)].reset_index(drop=True)
game_played3040

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,inhHqhkSY41yUqZBdTrfQzJNvLVDSTc3NP0CJ-pMXCWECi...,TOP,TOP,Kennen,15840.0,12.0,4.0,7.0,4.0,15.0,...,17.0,13200.0,425.212970,154.0,8.00,2.0,False,False,False,3103.0
1,421ERaCTa3XOWL-B37zDDq-fUgrHMka2nvlCSz04X90PD6...,JUNGLE,JUNGLE,Warwick,16570.0,11.0,4.0,9.0,6.0,16.0,...,33.0,13733.0,442.358738,178.0,4.00,1.0,False,False,False,3103.0
2,3mf9yiACQN5B6KqQcYiH5L4BHd0mxzm57LYwTf7BepcW74...,MIDDLE,MIDDLE,Zoe,15822.0,4.0,14.0,6.0,5.0,13.0,...,13.0,12318.0,396.790523,197.0,35.00,1.0,False,False,False,3103.0
3,nw03aFpUmsC5SMigTAjfFem779ViT1twLf-otdfWRn4WZk...,BOTTOM,BOTTOM,Samira,14743.0,7.0,4.0,16.0,5.0,10.0,...,18.0,16028.0,516.303129,223.0,7.00,2.0,False,False,False,3103.0
4,fzMBWWb-1HOBVc18kl8RywiuBVlcX9ImqtapnaaAOJCTQH...,UTILITY,UTILITY,Pyke,12659.0,3.0,4.0,5.0,4.0,15.0,...,19.0,9854.0,317.418470,46.0,27.00,1.0,False,False,False,3103.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Kayle,15748.0,4.0,12.0,2.0,2.0,10.0,...,16.0,11278.0,340.080333,190.0,1.00,0.0,False,True,False,3315.0
246,3QjhP6LhMEM98jxz_bbQejmz1jW1kWoUc9Bgk-nJ-zRTD8...,JUNGLE,JUNGLE,MonkeyKing,18282.0,11.0,4.0,6.0,1.0,5.0,...,25.0,12889.0,388.663675,214.0,52.35,3.0,False,True,False,3315.0
247,w23nDK6_Wg3SJuCz9dtfsUKjzIxm9nh-LRfA_KVLpcmL-K...,MIDDLE,MIDDLE,Irelia,18111.0,14.0,4.0,13.0,9.0,4.0,...,30.0,16285.0,491.070299,255.0,5.25,2.0,False,True,False,3315.0
248,Ggajs4IGPc6MJFU9Lvh1Jn7X6l9yc0ewRCmME2isd5qMlZ...,BOTTOM,BOTTOM,Ezreal,14056.0,7.0,4.0,7.0,3.0,8.0,...,15.0,13485.0,406.644434,229.0,38.00,1.0,False,True,False,3315.0


#### 40분 이후에 종료된 게임

In [15]:
# 3게임이 존재한다. 
game_played40up = game_df.copy()[4000 < game_df['timePlayed']].reset_index(drop=True)
game_played40up

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,bfgqzrmaXZGCbjxTentmmx9oE0zQZStxudqHBhq2Tr-P1Z...,TOP,TOP,Gnar,22920.0,4.0,12.0,8.0,9.0,4.0,...,25.0,16780.0,371.373666,260.0,3.00,1.0,False,False,False,4518.0
1,4QXLzL5X_M_SmBqmwwvi9oaqWlttIs2hEgL5gJ4JmQmwOr...,JUNGLE,JUNGLE,Ekko,18605.0,11.0,4.0,8.0,8.0,5.0,...,29.0,16744.0,370.566238,257.0,48.80,1.0,False,False,False,4518.0
2,BNdmUcT9oJ2Sg2skosh8bOAjkZFYuRca5eTh-iZ04TAGIy...,MIDDLE,MIDDLE,Morgana,23110.0,14.0,4.0,10.0,9.0,4.0,...,18.0,18316.0,405.356782,298.0,70.00,2.0,False,False,False,4518.0
3,MycXJK8eguKptKp5UyXUpDBMejQlLEi_OmB-65f4FcP1pl...,BOTTOM,BOTTOM,Sivir,18896.0,4.0,7.0,9.0,10.0,10.0,...,19.0,17977.0,397.855137,331.0,103.00,2.0,False,False,False,4518.0
4,7T6rf8SKLRoP6ckqjyi04Ri4r7Li7K67Pn9HjgEyXsHJk_...,UTILITY,UTILITY,Yuumi,15714.0,14.0,3.0,1.0,7.0,17.0,...,9.0,9978.0,220.838147,7.0,0.00,2.0,False,False,False,4518.0
5,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,27861.0,4.0,12.0,3.0,8.0,11.0,...,17.0,19136.0,423.512886,334.0,82.00,1.0,False,False,False,4518.0
6,HDiuYp4rmZKyheafff0TLkdtR2H6kooEESKsHUjTqsLiSw...,JUNGLE,JUNGLE,MonkeyKing,26148.0,11.0,4.0,14.0,1.0,21.0,...,27.0,21251.0,470.322088,226.0,9.20,2.0,False,False,False,4518.0
7,FHmCGRV_Ynp4ll5ywZjIY1ibeDtDhJ6ulI9F9yXvIRBRLJ...,MIDDLE,MIDDLE,Malzahar,23919.0,14.0,4.0,4.0,9.0,18.0,...,16.0,16599.0,367.366092,243.0,1.00,1.0,False,False,False,4518.0
8,NVlhW1BDQ2Z-huz-hOjgW9hhZLb5QyxZovScqZX-SOh6Vv...,BOTTOM,BOTTOM,Zeri,24173.0,7.0,4.0,14.0,10.0,13.0,...,23.0,20150.0,445.952247,276.0,7.00,1.0,False,False,False,4518.0
9,UTNOERaUNluxg7TB4HEWug5TelmeCpQL952bO47ewluRgb...,UTILITY,UTILITY,Pantheon,18661.0,4.0,14.0,8.0,8.0,18.0,...,17.0,14444.0,319.669389,70.0,64.00,2.0,False,False,False,4518.0


### 종료된 시간 대별, 라인별 평균 딜량, 챔피언 경험치, 골드획득을 확인한다. 

#### 20분 이내 종료된 게임의 라인별 평균 딜량, 챔피언 경험치, 골드 획득 확인

In [32]:
# 물론 트롤들과, 부캐, 대리, 이상치들이 섞인 평균이나 표본자체가 적어서 그대로 진행하였다. 
# 초반의 경우에는 템이 나오기전의 원딜, 라인전을 하지 않는 정글, 서포터 챔프의 특성으로 인해 
# 탑이나 미드라인에 비해서 딜량이 적게 나오는 것을 확인할 수 있다. 
# 챔피언 경험치 또한 2명이서 같이 받아먹는 봇라인의경우 적게 나오며 탑, 미드 라이너들의 솔로라인 경험치가 높다. 
# 골드획득의 경우에는 서폿을 제외한 모든 라인이 비슷한 것을 확인한다. 
# 20분 이전의 champExp 계산시 BOTTOM라인은 110% Middle라인은 90%, TOP 85%, UTILITY 130%
# 20분 이전의 totalDamageDealtToCHamp 계산시 JUNGLE 105%, MIDDLE 75%, TOP 90%, UTILITY 155%
# 20분 이전의 goldEarned 계산시 UTILITY 135%
# 위와 같은 비율로 평균화를 해주기로 하였다. 
game_played20down.groupby('teamPosition').mean()[['champExp', 'totalDamageDealtToChamp', 'goldEarned', 'damagePerMin', 'goldPerMin']]

,champExp,totalDamageDealtToChamp,goldEarned,damagePerMin,goldPerMin
teamPosition,,,,,
BOTTOM,5996.083333,6219.083333,6452.500000,366.539058,382.710818
JUNGLE,6655.333333,5626.166667,6138.166667,329.180428,361.286219
MIDDLE,7583.500000,8134.833333,6058.000000,471.270032,358.478756
TOP,7956.916667,7358.666667,6073.416667,440.663918,360.181782
UTILITY,5018.333333,3882.250000,4435.916667,221.788950,262.148677


#### 20분 이상, 30분 이내 종료된 게임의 라인별 평균 딜량, 챔피언 경험치, 골드 획득 확인

In [17]:
# 20분 이상 종료된 게임에서는 라인전 단계가 끝나고 한타, 오브젝트 싸움, 타워등에 있어서 경험치, 골드의 격차가 더 커질 것이다
# 20분에서 30분이하의 게임에서도 20분과 같이 여전히 바텀과 서폿의 경험치는 상체 라인에 비해서 적은걸로 확인되었다.
# 20분 이상 30분 이내 게임에서 champExp Bottom 110%, UTILIY 125%
# totalDamageDealtToChamp의 경우에는 정글의 라인전 갱킹, 등 여러한 부분에서 떨어지므로 딜량이 적게 나온다. 서폿또한 마찬가지이다.
# JUNGLE 105%, UTILITY 140% 
# goldEarned 에서는 UTILITY의 부분만 조정을 하면 된다. 130%
game_played2030.groupby('teamPosition').mean()[['champExp', 'totalDamageDealtToChamp', 'goldEarned']]

,champExp,totalDamageDealtToChamp,goldEarned
teamPosition,,,
BOTTOM,9831.743243,14589.229730,10045.472973
JUNGLE,10527.905405,12784.459459,9758.108108
MIDDLE,11369.567568,14806.986486,9461.310811
TOP,11748.500000,14169.810811,9327.702703
UTILITY,8432.445946,9106.689189,7141.662162


#### 30분 ~ 40분 사이의 종료된 게임의 라인별 평균 딜량, 챔피언 경험치, 골드 획득 확인하기

In [18]:
# 30~40분 사이의 게임에서는 한타, 오브젝트 관리에 대해서 가장 중요한 시기이다. 또한 스플릿 등등 라인 관리에 대해서도 중요하다. 
# 게임의 딜량에 대해서는 원딜, 미드, 탑에 의존하는 경우가 높아진다. 
# 챔피언 경험치에 대해서는 원딜의 솔라인으로 인해서 정글과 비슷해지며 탑의 경우에는 텔포를 바탕으로 스플릿을 진행하여 여전히 높다. 
# champEXP MIDDLE 95%, TOP 90%, UTILITY 125% 
# totalDamageDealtToCHamp BOTTOM 85%, MIDDLE 85%, TOP 95%, UTILITY 150% 
# goldEARNED UTILITY 140% 
game_played3040.groupby('teamPosition').mean()[['champExp', 'totalDamageDealtToChamp', 'goldEarned']]

,champExp,totalDamageDealtToChamp,goldEarned
teamPosition,,,
BOTTOM,15866.80,26456.58,14977.02
JUNGLE,16045.44,22471.58,13898.44
MIDDLE,16852.94,25952.48,13831.22
TOP,17675.80,24704.16,13874.06
UTILITY,12828.24,15808.66,9837.42


#### 40분 이상 종료된 게임의 라인별 딜량, 골드 획득, 경험치 분석

In [19]:
# 40분 이상의 경우에는 원딜, 미드의 딜량이 가장 중요하며 탑의 스플릿, 바론, 장로와 같은 오브젝트 싸움 한번에 게임이 끝난다.
# 원딜의 경우에는 탑, 정글, 미드의 집중으로 인해서 폭사하는 경우도 생기며 끝까지 살아서 딜을 폭발시키기도 한다.
# 최대 경험치에 이르는 경우도 많다. 
# champExp UTILITY 120%
# total DamageDealtToChamp JUNGLE 110%, MIDDLE 90%, UTILITY 110%
# goldEarned BOTTOM 90%, MIDDLE 90%, UTILITY 125%
game_played40up.groupby('teamPosition').mean()[['champExp', 'totalDamageDealtToChamp', 'goldEarned']]

,champExp,totalDamageDealtToChamp,goldEarned
teamPosition,,,
BOTTOM,21549.250,35932.750,18705.250
JUNGLE,21606.875,31228.250,17481.625
MIDDLE,22774.875,38563.375,18139.375
TOP,23313.500,33734.250,17676.000
UTILITY,17905.125,29230.750,13283.500


## 분석 데이터프레임 수정하기

In [20]:
game_df[(game_df['timePlayed'] < 2000) & (game_df['teamPosition']== 'TOP')]

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
90,B6f-hh1my-rb1rAAAQTgeJGKVcSXZxZpVPRdWWsUvVeVDO...,TOP,TOP,Nilah,6025.0,4.0,12.0,1.0,3.0,0.0,...,29.0,4026.0,296.781612,88.0,7.0,1.0,False,True,False,1355.0
95,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,7597.0,4.0,12.0,3.0,1.0,1.0,...,24.0,5326.0,392.643939,96.0,8.0,2.0,False,True,False,1355.0
160,9yf8Rs_z-wYgmGrOEt3c5Dxt5vxXciWjSutOB3cJe_X5_b...,TOP,TOP,Jax,6558.0,12.0,4.0,2.0,7.0,0.0,...,30.0,5036.0,320.051088,80.0,1.0,0.0,False,True,False,1573.0
165,qgPQotn5woPPNGVWHTMKkNsxSO15eUKgKArQWn2D06bxpW...,TOP,TOP,Darius,8809.0,4.0,6.0,8.0,2.0,0.0,...,31.0,7298.0,463.812662,117.0,45.0,3.0,False,True,False,1573.0
190,yKZnzbQCJcT-VAKRYJSFdo_wn8SGREc6G4yiCOdiRRkJEH...,TOP,TOP,Darius,6800.0,6.0,4.0,2.0,1.0,0.0,...,15.0,5289.0,347.043530,116.0,38.0,1.0,False,True,False,1523.0
195,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Akali,6456.0,4.0,12.0,0.0,2.0,2.0,...,17.0,4409.0,289.324512,87.0,0.0,1.0,False,True,False,1523.0
230,nxp0FYb8Bm2X7Is7P_8t_LytQBIIU-48MqPrBgruO4dVeX...,TOP,TOP,Kayle,10480.0,12.0,4.0,2.0,1.0,0.0,...,16.0,7990.0,409.279655,154.0,36.0,1.0,False,True,False,1952.0
235,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,8837.0,4.0,12.0,1.0,2.0,0.0,...,20.0,5923.0,303.391759,118.0,7.0,1.0,False,True,False,1952.0
700,D58qSu1hYSOnEBJs74qZ5VCt6PymDZOnoCGErGYRUz_NWz...,TOP,TOP,Soraka,7692.0,12.0,4.0,2.0,4.0,7.0,...,17.0,5395.0,285.634450,68.0,7.0,2.0,False,True,False,1888.0
705,5QiWucuMelUIjTolOPSi2MLBaq1br8iHA6cEbw84cKQAww...,TOP,TOP,Tristana,6979.0,4.0,14.0,4.0,4.0,0.0,...,15.0,6811.0,360.623341,100.0,46.0,2.0,False,True,False,1888.0


In [21]:
game_df.loc[11]

puuid                           4QXLzL5X_M_SmBqmwwvi9oaqWlttIs2hEgL5gJ4JmQmwOr...
teamPosition                                                               JUNGLE
individualPosition                                                         JUNGLE
champName                                                                    Ekko
champExp                                                                  18605.0
summoner_spell1                                                              11.0
summoner_spell2                                                               4.0
kills                                                                         8.0
deaths                                                                        8.0
assists                                                                       5.0
damageDealtToBuildings                                                     1358.0
totalDamageDealtToChamp                                                   30012.0
damagePerMin    

In [22]:
game_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   puuid                         720 non-null    object 
 1   teamPosition                  720 non-null    object 
 2   individualPosition            720 non-null    object 
 3   champName                     720 non-null    object 
 4   champExp                      720 non-null    float64
 5   summoner_spell1               720 non-null    float64
 6   summoner_spell2               720 non-null    float64
 7   kills                         720 non-null    float64
 8   deaths                        720 non-null    float64
 9   assists                       720 non-null    float64
 10  damageDealtToBuildings        720 non-null    float64
 11  totalDamageDealtToChamp       720 non-null    float64
 12  damagePerMin                  720 non-null    float64
 13  teamD

In [23]:
game_df.describe()

,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,damageDealtToBuildings,totalDamageDealtToChamp,damagePerMin,teamDamagePer(%),killParticipation(%),totalDamageTaken,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,timePlayed
count,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.00000,720.000000,720.000000,720.000000
mean,12584.072222,8.815278,6.068056,5.472222,5.484722,7.475000,2517.038889,17135.654167,571.359968,19.998611,46.754219,22012.298611,20.006944,10746.423611,371.680290,142.17500,25.325972,1.509722,2872.513889
std,4488.072167,4.389967,3.610305,4.444910,3.207222,5.167776,2845.144969,10492.733992,269.760540,7.675349,15.964968,11230.802874,5.914430,3866.175418,86.822634,75.17836,29.986941,0.953678,736.768779
min,3038.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1243.000000,70.375358,3.000000,0.000000,1419.000000,5.000000,2442.000000,169.301926,1.00000,0.000000,0.000000,1355.000000
25%,9269.000000,4.000000,4.000000,2.000000,3.000000,4.000000,390.000000,9496.500000,373.175841,14.000000,36.646341,14138.250000,16.000000,7882.000000,308.612711,93.75000,4.000000,1.000000,2301.500000
50%,12064.500000,11.000000,4.000000,4.000000,5.000000,7.000000,1650.500000,14629.000000,534.384242,20.000000,47.673160,19854.000000,20.000000,10215.500000,364.832433,146.00000,13.000000,1.000000,2775.000000
75%,15514.500000,14.000000,7.000000,8.000000,8.000000,10.000000,3734.000000,22728.000000,726.331034,25.000000,57.142857,27829.500000,24.000000,13244.500000,433.272945,194.00000,37.000000,2.000000,3399.750000
max,27861.000000,21.000000,21.000000,26.000000,17.000000,27.000000,25514.000000,66793.000000,1768.984191,44.000000,100.000000,80835.000000,38.000000,23339.000000,700.094282,378.00000,222.500000,6.000000,4815.000000


In [24]:
game_df.loc[10]['champExp']

22920.0

In [25]:
# 위에서 분석한 시간대별 골드획득, 적에게 가한 피해량, 경험치 획득 지표를 통해서 값 변경해주기
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

for i in range(len(game_df)):
    if game_df.loc[i, 'timePlayed'] < 2000:

        if game_df.loc[i, 'teamPosition'] == 'TOP':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 0.85
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.9
            
        elif game_df.loc[i, 'teamPosition'] == 'JUNGLE':
             game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.05
            
        elif game_df.loc[i, 'teamPosition'] == 'MIDDLE':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 0.9
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.75
            
        elif game_df.loc[i, 'teamPosition'] == 'BOTTOM':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.1

        elif game_df.loc[i, 'teamPosition'] == 'UTILITY':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.3
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.55
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 1.35
            
        else:
            pass
    elif (2000 <= game_df.loc[i, 'timePlayed']) & (game_df.loc[i, 'timePlayed'] < 3000):      
        if game_df.loc[i,  'teamPosition'] == 'JUNGLE':
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.05
        
        elif game_df.loc[i, 'teamPosition']== 'BOTTOM':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.1


        elif game_df.loc[i, 'teamPosition'] == 'UTILITY':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.25
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.4
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 1.3
            
        else:
            pass
    
    elif (3000 <= game_df.loc[i,'timePlayed']) & (game_df.loc[i, 'timePlayed'] < 4000):

        if game_df.loc[i, 'teamPosition'] == 'TOP':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 0.9
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] *0.85
        
        elif game_df.loc[i, 'teamPosition'] == 'MIDDLE':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 0.95
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.85

        elif game_df.loc[i, 'teamPosition'] == 'BOTTOM':
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.85

        elif game_df.loc[i, 'teamPosition'] == 'UTILITY':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.25
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 1.5
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 1.4 
    elif game_df.loc[i, 'timePlayed'] >=4000:
        
        if game_df.loc[i, 'teamPosition'] == 'JUNGGLE':
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] *1.1
        
        elif game_df.loc[i, 'teamPosition'] == 'MIDDLE':
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] * 0.9
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 0.9
        
        elif game_df.loc[i, 'teamPosition'] == 'BOTTOM':
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 0.9

        elif game_df.loc[i, 'teamPosition'] == 'UTILITY':
            game_df.loc[i, 'champExp'] = game_df.loc[i, 'champExp'] * 1.2
            game_df.loc[i, 'totalDamageDealtToChamp'] = game_df.loc[i, 'totalDamageDealtToChamp'] *1.1
            game_df.loc[i, 'goldEarned'] = game_df.loc[i, 'goldEarned'] * 1.25

In [26]:
# 가장 최신 게임에 대해서 확인해보자면 
# 트롤은 미스포츈, 모르가나였다. 바텀에서 분열로 인해서 게임이 힘들었지만
# 상체 3명의 캐리로 인해서 게임을 이긴 경기
# 확인해보자면 챔피언 경험치는 미스포츈이 팀에서 제일 낮게 나오며, 
# 팀내 데미지 비중, 적에게 가한 피해량, 골드 획득, 분당 골드획득
# 상대 라인보다 많은 CS, 레벨 전부 없었다. 
game_df.loc[0:9]

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,damageTakeOnTeamPer(%),goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timePlayed
0,inhHqhkSY41yUqZBdTrfQzJNvLVDSTc3NP0CJ-pMXCWECi...,TOP,TOP,Kennen,14256.00,12.0,4.0,7.0,4.0,15.0,...,17.0,13200.0,425.212970,154.0,8.00,2.0,False,False,False,3103.0
1,421ERaCTa3XOWL-B37zDDq-fUgrHMka2nvlCSz04X90PD6...,JUNGLE,JUNGLE,Warwick,16570.00,11.0,4.0,9.0,6.0,16.0,...,33.0,13733.0,442.358738,178.0,4.00,1.0,False,False,False,3103.0
2,3mf9yiACQN5B6KqQcYiH5L4BHd0mxzm57LYwTf7BepcW74...,MIDDLE,MIDDLE,Zoe,15030.90,4.0,14.0,6.0,5.0,13.0,...,13.0,12318.0,396.790523,197.0,35.00,1.0,False,False,False,3103.0
3,nw03aFpUmsC5SMigTAjfFem779ViT1twLf-otdfWRn4WZk...,BOTTOM,BOTTOM,Samira,14743.00,7.0,4.0,16.0,5.0,10.0,...,18.0,16028.0,516.303129,223.0,7.00,2.0,False,False,False,3103.0
4,fzMBWWb-1HOBVc18kl8RywiuBVlcX9ImqtapnaaAOJCTQH...,UTILITY,UTILITY,Pyke,15823.75,3.0,4.0,5.0,4.0,15.0,...,19.0,13795.6,317.418470,46.0,27.00,1.0,False,False,False,3103.0
5,HSVz53y3wDCfvKxZtwu8HTXZSRJbWIsQf7XQqKjoQUVPrp...,TOP,TOP,Darius,12300.30,6.0,4.0,5.0,10.0,6.0,...,22.0,11333.0,365.082180,183.0,34.00,1.0,False,False,False,3103.0
6,SUQnKQ9C6yC_8Ysykj65iEBWuiDGz2MCkc3kxmF0nzhGNv...,JUNGLE,JUNGLE,Trundle,15519.00,4.0,11.0,8.0,5.0,7.0,...,30.0,14288.0,460.266535,200.0,34.05,1.0,False,False,False,3103.0
7,qA5YgCfHkO-NrdUR_9cQ2-te4NhLNzKddJmC0cDx7iLwds...,MIDDLE,MIDDLE,Lux,13189.80,4.0,12.0,6.0,9.0,2.0,...,15.0,10655.0,343.234416,172.0,8.00,2.0,False,False,False,3103.0
8,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,BOTTOM,BOTTOM,Twitch,12249.00,4.0,7.0,3.0,11.0,9.0,...,19.0,10826.0,348.730464,195.0,21.00,1.0,False,False,False,3103.0
9,bMkcHlsxML9gRVD3re35MgKokWiBMaRZ5wUTMM3ZtapSkC...,UTILITY,UTILITY,Renata,13980.00,14.0,4.0,2.0,8.0,12.0,...,14.0,11316.2,260.373796,22.0,0.00,1.0,False,False,False,3103.0


In [27]:
game_df.columns

Index(['puuid', 'teamPosition', 'individualPosition', 'champName', 'champExp',
       'summoner_spell1', 'summoner_spell2', 'kills', 'deaths', 'assists',
       'damageDealtToBuildings', 'totalDamageDealtToChamp', 'damagePerMin',
       'teamDamagePer(%)', 'killParticipation(%)', 'totalDamageTaken',
       'damageTakeOnTeamPer(%)', 'goldEarned', 'goldPerMin', 'totalCs',
       'maxCsAdvantageOnLaneOpponent', 'maxLevelLeadLaneOpponent',
       'gameEndedInEarlySurren', 'gameEndedInSurren', 'teamEarlySurrn',
       'timePlayed'],
      dtype='object')

In [30]:
game_df['damagePerMin'][0:10]

0     978.948202
1     715.462307
2     848.981215
3    1020.327474
4     329.491624
5     412.294617
6     721.822334
7     725.138081
8     840.924938
9     271.772998
Name: damagePerMin, dtype: float64

In [37]:
game_df.columns

Index(['puuid', 'teamPosition', 'individualPosition', 'champName', 'champExp',
       'summoner_spell1', 'summoner_spell2', 'kills', 'deaths', 'assists',
       'damageDealtToBuildings', 'totalDamageDealtToChamp', 'damagePerMin',
       'teamDamagePer(%)', 'killParticipation(%)', 'totalDamageTaken',
       'damageTakeOnTeamPer(%)', 'goldEarned', 'goldPerMin', 'totalCs',
       'maxCsAdvantageOnLaneOpponent', 'maxLevelLeadLaneOpponent',
       'gameEndedInEarlySurren', 'gameEndedInSurren', 'teamEarlySurrn',
       'timePlayed'],
      dtype='object')

In [39]:
# 새롭게 0인 칼럼을 추가한다. 
# 이 컬럼은 10명의 플레이어에 대해서 지표를 통해서 순위를 매기는 방식
# 게임 경험치, 적에게 가한 피해량, 골드획득은 높을수록 좋은 지표 등수로는 높은순부터 1
# maxCsAdvantageOnLaneOpponent, maxLevelLeadLaneOpponent또한 높을수록 좋은 지표, 등수로는 높은순부터 1
# teamdamagePer에 대해서는 좀 지표세우기가 힘들다. 트롤로써 많이 죽어서 높을수도 있고 탱커라는 역활에 따라서 높을수도 있다. 

game_df['rank'] = 0
for i in range(0, len(game_df), 10):
    one_game_df = game_df[i : i+10]
    game_df['rank'].loc[i:i+10] += one_game_df['champExp'].rank(ascending=False, method='min')
    game_df['rank'].loc[i:i+10] += one_game_df['totalDamageDealtToChamp'].rank(ascending=False, method='min')
    game_df['rank'].loc[i:i+10] += one_game_df['goldEarned'].rank(ascending=False, method='min')
    game_df['rank'].loc[i:i+10] += one_game_df['maxCsAdvantageOnLaneOpponent'].rank(ascending=False, method='min')
    game_df['rank'].loc[i:i+10] += one_game_df['maxLevelLeadLaneOpponent'].rank(ascending=False, method='min')


C:\Users\lima\AppData\Local\Temp\ipykernel_15216\1985507746.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  game_df['rank'].loc[i:i+10] += one_game_df['champExp'].rank(ascending=False, method='min')


In [61]:
# 1번째 게임에서 5개의 지표중 평균 7.5등을 기록한 플레이어는 
# 레나타라는 플레이어다. 레나타 플레이어에 대해서 실제 게임 타임라인을 통해 트롤여부를 확실히 판단하자
max(game_df['rank'][0:10])

38.0

In [62]:
# 게임 타임라인 불러오기
def game_timeline(match_id_num):
    game_timeline_url = 'https://asia.api.riotgames.com/lol/match/v5/matches/'+match_id_num+'/timeline'
    return requests.get(game_timeline_url, headers=request_headers).json()

In [66]:
aa = game_timeline(match_id[0])

In [94]:
aa['info']['frames'][0].keys()

dict_keys(['events', 'participantFrames', 'timestamp'])

In [96]:
aa['info']['frames'][0]['events']

[{'realTimestamp': 1658211478245, 'timestamp': 0, 'type': 'PAUSE_END'}]

In [109]:
aa['info']['frames'][32]['timestamp']

1862702

In [126]:
len(aa['info']['frames'])

33

In [127]:
aa['info'].keys()

dict_keys(['frameInterval', 'frames', 'gameId', 'participants'])

In [129]:
len(aa['info']['frames'])

33

In [131]:
aa['info']['frames'][0].keys()

dict_keys(['events', 'participantFrames', 'timestamp'])

In [135]:
len(aa['info']['frames'][1]['events'])

30

In [156]:
aa['info']['frames'][5]['events'][11]

{'creatorId': 2,
 'timestamp': 249196,
 'type': 'WARD_PLACED',
 'wardType': 'UNDEFINED'}

In [ ]:
game_timeline_df = pd.DataFrame(columns= ['participantId', 'level', 'totalGold', 'xp','armor', 'attackDamage', 'attackSpeed', 'health', 'magicResist', 'totalDamageDoneToChamp', 'goldPerSecond', 'level', 'totalCs'])

In [165]:
for i in range(len(aa['info']['frames'])):
    

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32


In [168]:
aa['info']['frames'][0]['participantFrames'][str(1)]

{'championStats': {'abilityHaste': 0,
  'abilityPower': 0,
  'armor': 29,
  'armorPen': 0,
  'armorPenPercent': 0,
  'attackDamage': 25,
  'attackSpeed': 100,
  'bonusArmorPenPercent': 0,
  'bonusMagicPenPercent': 0,
  'ccReduction': 0,
  'cooldownReduction': 0,
  'health': 611,
  'healthMax': 611,
  'healthRegen': 0,
  'lifesteal': 0,
  'magicPen': 0,
  'magicPenPercent': 0,
  'magicResist': 30,
  'movementSpeed': 335,
  'omnivamp': 0,
  'physicalVamp': 0,
  'power': 200,
  'powerMax': 200,
  'powerRegen': 0,
  'spellVamp': 0},
 'currentGold': 500,
 'damageStats': {'magicDamageDone': 0,
  'magicDamageDoneToChampions': 0,
  'magicDamageTaken': 0,
  'physicalDamageDone': 0,
  'physicalDamageDoneToChampions': 0,
  'physicalDamageTaken': 0,
  'totalDamageDone': 0,
  'totalDamageDoneToChampions': 0,
  'totalDamageTaken': 0,
  'trueDamageDone': 0,
  'trueDamageDoneToChampions': 0,
  'trueDamageTaken': 0},
 'goldPerSecond': 0,
 'jungleMinionsKilled': 0,
 'level': 1,
 'minionsKilled': 0,
 'pa